# 03 · Medidas e valores de referência

**Objetivo.** Calcular, uma vez e em um só lugar, **cada medida que o dashboard vai exibir**, com o
valor que a medida DAX correspondente terá de devolver. Enquanto o número existir apenas dentro do
`.pbix`, não há como saber se um filtro implícito, um relacionamento ou um `DISTINCTCOUNT` mal
escolhido mudou o resultado. Com o gabarito fora do arquivo, a conferência é aritmética: ou o
Power BI devolve o mesmo número, ou a medida está errada.

**Entradas**

| Arquivo | Papel aqui |
|---|---|
| `00_data/bronze/amazon.csv` | **somente leitura** — apenas a contagem de linhas da origem (1.465) |
| `00_data/silver/produtos.csv` | base de produtos (1.351) e as duas bases intermediárias de validade |
| `00_data/gold/base_analitica.csv` | base analítica (1.348) — padrão de toda medida cruzada |
| `00_data/gold/reviews.csv` | corpus deduplicado por `review_id` (9.255) |

**Saídas**

| Arquivo | Conteúdo |
|---|---|
| `02_results/tables/tab12_valores_referencia.csv` | o gabarito do DAX — `medida · valor · base · n · observacao` |
| `02_results/images/fig_med_01_painel_medidas.png` | painel de conferência das contagens categóricas |
| `02_results/images/fig_med_02_participacao_categoria.png` | participação de cada categoria no volume de avaliações |

**Regras transversais aplicadas neste notebook**

- **T1** — a nota agregada publicada é a `nota_corrigida` (limite inferior de Wilson), sempre com o *n* ao lado.
- **T2** — nenhum grupo agregado com *n* < 30 é ranqueado ou comparado; as 5 categorias abaixo do piso
  aparecem somadas em uma linha declarada como suprimida, nunca lado a lado com as outras quatro.
- **T3** — toda linha do gabarito declara a **base** e o ***n***; toda figura repete isso no rodapé.
- **T4** — este notebook mede, não explica. Nenhuma linha afirma causa.
- **T6** — `rating_count` é **volume de avaliações**. A palavra "venda", "receita", "demanda" ou
  "popularidade" não aparece em nenhum rótulo daqui.
- **T7** — `user_id` e `user_name` não existem a partir da camada silver; nada a exibir.
- **T8** — as medidas de corpus declaram a unidade, deduplicam por `review_id` e citam o truncamento em 8.
- **T9** — a paleta v06 é aplicada por módulo, na primeira célula de código. Nenhuma cor escolhida à mão.
- **Moeda (L5)** — valores monetários em **rupia indiana, sem símbolo e sem conversão**.

## 1. Paleta e ambiente

Primeira célula de código do notebook, sempre. A paleta v06 é aplicada por ferramenta (`src/paleta.py`),
nunca por escolha de cor no gráfico — é o que T9 exige. Se esta célula não rodar, as figuras saem na
paleta padrão do matplotlib e o notebook está em descumprimento mesmo que todos os números estejam certos.

In [1]:
import sys; sys.path.insert(0, "../src")
from paleta import rodape, aplicar, confere_visual
aplicar()

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from paleta import CATEGORICA, CHAMADAS, ESTRUTURA, BASE, checar_series

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

RAIZ = os.path.abspath("..")
IMG = os.path.join(RAIZ, "02_results", "images")
TAB = os.path.join(RAIZ, "02_results", "tables")
os.makedirs(IMG, exist_ok=True)
os.makedirs(TAB, exist_ok=True)

AMBAR, AZUL, CLARA = CATEGORICA[0], CATEGORICA[1], CATEGORICA[2]
LARANJA, TEAL, ARDOSIA = CHAMADAS["atencao"], CHAMADAS["favoravel"], ESTRUTURA["secundario"]
TINTA, SECUNDARIO = ESTRUTURA["tinta"], ESTRUTURA["secundario"]

print("paleta v06 aplicada ·", len(CATEGORICA), "cores categóricas")

paleta v06 aplicada · 6 cores categóricas


## 2. As bases, e por que cada medida precisa dizer em qual delas nasceu

Cinco bases convivem no projeto. Duas medidas corretas podem discordar apenas por terem sido contadas
em bases diferentes — e é exatamente esse o erro que o gabarito precisa impedir. Por isso o registro de
cada medida carrega, obrigatoriamente, a base e o *n* (T3).

O bronze é aberto **apenas para contar linhas**. Ele não é escrito, não é filtrado e não alimenta
nenhuma medida: contém `user_id` e `user_name`, e T7 veda que esses campos cheguem a qualquer saída.

In [2]:
# --- bronze: só a contagem de linhas da origem. Nenhuma coluna é usada adiante. ---
CAMINHO_BRONZE = os.path.join(RAIZ, "00_data", "bronze", "amazon.csv")
try:
    n_bruta = len(pd.read_csv(CAMINHO_BRONZE, dtype=str, usecols=["product_id"]))
    origem_bruta = "medido em 00_data/bronze/amazon.csv"
except FileNotFoundError:
    # O bronze não é versionado (T7/LGPD). Sem ele, cai-se no valor declarado no material.
    n_bruta = 1465
    origem_bruta = "bronze ausente — valor declarado em method/definitions/base_factual_e_bases_de_calculo.md"
print(f"base bruta: {n_bruta} linhas · {origem_bruta}")

# --- silver: produtos deduplicados e tipados, já sem colunas pessoais ---
silver = pd.read_csv(os.path.join(RAIZ, "00_data", "silver", "produtos.csv"))
n_produtos = len(silver)
n_nota_valida = int(silver.nota.notna().sum())
n_engaj_valido = int(silver.avaliacoes.notna().sum())

# --- gold: a base analítica, padrão de toda medida cruzada ---
gold = pd.read_csv(os.path.join(RAIZ, "00_data", "gold", "base_analitica.csv"))
n_analitica = len(gold)

# --- gold: corpus de avaliações, já deduplicado por review_id (T8) ---
reviews = pd.read_csv(os.path.join(RAIZ, "00_data", "gold", "reviews.csv"))
n_reviews = len(reviews)

assert "user_id" not in silver.columns and "user_name" not in silver.columns, "T7 violada na silver"
assert "user_id" not in gold.columns and "user_name" not in gold.columns, "T7 violada na gold"

print(f"produtos (silver)        : {n_produtos}")
print(f"nota válida              : {n_nota_valida}")
print(f"engajamento válido       : {n_engaj_valido}")
print(f"analítica (gold)         : {n_analitica}")
print(f"reviews distintos (gold) : {n_reviews}")

base bruta: 1465 linhas · medido em 00_data/bronze/amazon.csv
produtos (silver)        : 1351
nota válida              : 1350
engajamento válido       : 1349
analítica (gold)         : 1348
reviews distintos (gold) : 9255


## 3. O registro do gabarito

Toda medida daqui em diante entra na lista por uma única função. Não há número solto no notebook: se
não foi registrado, não vai para o CSV — e se não vai para o CSV, não pode ser cobrado do DAX.

O campo `valor` é **numérico e cru** (ponto decimal, sem separador de milhar e sem símbolo de moeda),
porque é ele que será comparado com o retorno da medida no Power BI. A formatação pt-BR é problema de
apresentação, não de conferência. A unidade vive no nome da medida; a regra de cálculo, na observação.

In [3]:
REFERENCIA = []


# Acrescenta uma linha ao gabarito. Base e n são obrigatórios — é o que T3 exige.
def registrar(medida, valor, base, n, observacao):
    if base is None or n is None:
        raise ValueError(f"'{medida}' sem base ou sem n declarado — T3 não permite publicar.")
    REFERENCIA.append({
        "medida": medida,
        "valor": round(float(valor), 4),
        "base": base,
        "n": int(n),
        "observacao": observacao,
    })
    return REFERENCIA[-1]


# ---- Bloco 1: contagem de produtos por base -------------------------------
registrar("Contagem de linhas — base bruta", n_bruta, "Bruta", n_bruta,
          "linhas da origem, antes da deduplicação; serve só para relatar a qualidade do arquivo")
registrar("Contagem de produtos — base de produtos", n_produtos, "Produtos", n_produtos,
          "DISTINCTCOUNT(product_id) após a regra de deduplicação; 1.465 − 114 linhas excedentes")
registrar("Contagem de produtos — nota válida", n_nota_valida, "Nota válida", n_nota_valida,
          "produtos com rating convertível; 1 registro com valor '|' vira nulo e sai")
registrar("Contagem de produtos — engajamento válido", n_engaj_valido, "Engajamento válido", n_engaj_valido,
          "produtos com rating_count convertível; 2 nulos saem, sem sobreposição com o anterior")
registrar("Contagem de produtos — base analítica", n_analitica, "Analítica", n_analitica,
          "padrão do dashboard: nota e volume de avaliações simultaneamente válidos")
registrar("Contagem de avaliações distintas — corpus", n_reviews, "Corpus de avaliações", n_reviews,
          "T8: unidade = review individual, deduplicada por review_id; não confundir com contagem de linhas")

pd.DataFrame(REFERENCIA)

,medida,valor,base,n,observacao
0,Contagem de linhas — base bruta,1465.0,Bruta,1465,"linhas da origem, antes da deduplicação; serve..."
1,Contagem de produtos — base de produtos,1351.0,Produtos,1351,DISTINCTCOUNT(product_id) após a regra de dedu...
2,Contagem de produtos — nota válida,1350.0,Nota válida,1350,produtos com rating convertível; 1 registro co...
3,Contagem de produtos — engajamento válido,1349.0,Engajamento válido,1349,produtos com rating_count convertível; 2 nulos...
4,Contagem de produtos — base analítica,1348.0,Analítica,1348,padrão do dashboard: nota e volume de avaliaçõ...
5,Contagem de avaliações distintas — corpus,9255.0,Corpus de avaliações,9255,"T8: unidade = review individual, deduplicada p..."


## 4. Cortes operacionais

Os três cortes que sustentam quase todas as marcações do dashboard. Eles não são arbitrados aqui: são
quantis medidos sobre a base analítica. O DAX **não deve recalcular o quantil em tempo de filtro** — se
recalcular, o corte muda quando o usuário clica numa categoria, e dois visuais da mesma página passam a
classificar o mesmo produto de formas diferentes. O valor entra como constante, e é este.

In [4]:
p90_avaliacoes = float(gold.avaliacoes.quantile(0.90))
mediana_avaliacoes = float(gold.avaliacoes.median())

registrar("Corte de engajamento alto — p90 de avaliações", p90_avaliacoes, "Analítica", n_analitica,
          "constante no modelo; recalcular o quantil sob filtro faz o corte variar entre visuais")
registrar("Corte de engajamento baixo — mediana de avaliações", mediana_avaliacoes, "Analítica", n_analitica,
          "constante no modelo, pela mesma razão do p90")
registrar("Corte de satisfação baixa — nota", 4.0, "Analítica", n_analitica,
          "rating < 4,0; limiar fechado no material, não é quantil")
registrar("Corte de satisfação alta — nota", 4.3, "Analítica", n_analitica,
          "rating >= 4,3; limiar fechado no material, não é quantil")
registrar("Piso amostral de grupo agregado", 30, "Analítica", n_analitica,
          "T2: abaixo de 30 o grupo não é ranqueado nem comparado; listas nominais de produtos são exceção")

print(f"p90 de avaliações      : {p90_avaliacoes:,.0f}".replace(",", "."))
print(f"mediana de avaliações  : {mediana_avaliacoes:,.0f}".replace(",", "."))

p90 de avaliações      : 36.384
mediana de avaliações  : 4.740


## 5. Medidas de posição

Medianas, não médias. As três variáveis monetárias e o volume de avaliações são fortemente assimétricos
— o preço de tabela vai de 39 a 139.900 e as avaliações a 426.973 —, e a média de uma cauda dessas
descreve o extremo, não o portfólio. O dashboard exibe mediana; o gabarito registra mediana.

A `nota_corrigida` aparece ao lado da nota simples por T1: é ela que sustenta qualquer ordenação por nota.
Os dois números convivem no gabarito porque o cartão do dashboard mostra a nota simples e o ranking usa a
corrigida — e é previsível que alguém confunda os dois se só um estiver registrado.

Valores monetários em **rupia indiana, sem símbolo e sem conversão** (L5).

In [5]:
registrar("Preço de tabela mediano (rupia indiana)", gold.preco_tab.median(), "Analítica", n_analitica,
          "MEDIAN(preco_tab); sem símbolo e sem conversão — L5")
registrar("Preço com desconto mediano (rupia indiana)", gold.preco_desc.median(), "Analítica", n_analitica,
          "MEDIAN(preco_desc); é o preço praticado, não o de tabela")
registrar("Desconto mediano (%)", gold.desconto.median(), "Analítica", n_analitica,
          "MEDIAN(desconto), escala 0–100; divergência máxima de 0,50 p.p. contra o recalculado")
registrar("Nota mediana", gold.nota.median(), "Analítica", n_analitica,
          "MEDIAN(nota); exibição em cartão, nunca base de ranking — T1")
registrar("Nota corrigida mediana (Wilson)", gold.nota_corrigida.median(), "Analítica", n_analitica,
          "MEDIAN(nota_corrigida); coluna calculada na carga, não medida DAX — Wilson depende do n do produto")
registrar("Volume mediano de avaliações", gold.avaliacoes.median(), "Analítica", n_analitica,
          "MEDIAN(avaliacoes); T6 — volume de avaliações, nunca vendas ou demanda")
registrar("Volume total de avaliações", gold.avaliacoes.sum(), "Analítica", n_analitica,
          "SUM(avaliacoes); denominador de toda participação percentual deste gabarito")
registrar("Média de avaliações por produto", gold.avaliacoes.mean(), "Analítica", n_analitica,
          "registrada só para expor a assimetria contra a mediana; o dashboard não publica média")

posicao = pd.DataFrame(REFERENCIA[-8:])[["medida", "valor"]]
print(posicao.to_string(index=False))

                                    medida        valor
   Preço de tabela mediano (rupia indiana) 1.795000e+03
Preço com desconto mediano (rupia indiana) 8.990000e+02
                      Desconto mediano (%) 4.900000e+01
                              Nota mediana 4.100000e+00
           Nota corrigida mediana (Wilson) 4.065200e+00
              Volume mediano de avaliações 4.740000e+03
                Volume total de avaliações 2.380147e+07
           Média de avaliações por produto 1.765688e+04


## 6. Contagens por faixa de engajamento e de satisfação

São as duas segmentações categóricas do dashboard. Cada uma parte do total da base analítica, e é assim
que devem ser conferidas: a soma das faixas tem de fechar em 1.348, sem "não classificado". Qualquer
sobra indica produto fora dos cortes — e, portanto, uma medida DAX escrita com desigualdade trocada.

In [6]:
ORDEM_ENGAJAMENTO = ["alto", "intermediario", "baixo"]
ORDEM_SATISFACAO = ["alta", "media", "baixa"]

conta_engajamento = gold.engajamento.value_counts().reindex(ORDEM_ENGAJAMENTO)
conta_satisfacao = gold.satisfacao.value_counts().reindex(ORDEM_SATISFACAO)

criterio_engajamento = {
    "alto": f"avaliacoes >= {p90_avaliacoes:,.0f}".replace(",", ".") + " (p90)",
    "intermediario": "entre a mediana (inclusive) e o p90 (exclusive)",
    "baixo": f"avaliacoes < {mediana_avaliacoes:,.0f}".replace(",", ".") + " (mediana)",
}
criterio_satisfacao = {
    "alta": "nota >= 4,3", "media": "nota entre 4,0 e 4,3", "baixa": "nota < 4,0",
}

for faixa in ORDEM_ENGAJAMENTO:
    registrar(f"Produtos com engajamento {faixa}", int(conta_engajamento[faixa]), "Analítica", n_analitica,
              criterio_engajamento[faixa] + " · T6: volume de avaliações")
for faixa in ORDEM_SATISFACAO:
    registrar(f"Produtos com satisfação {faixa}", int(conta_satisfacao[faixa]), "Analítica", n_analitica,
              criterio_satisfacao[faixa])

registrar("Participação de satisfação baixa no portfólio (%)",
          100 * conta_satisfacao["baixa"] / n_analitica, "Analítica", n_analitica,
          "nota < 4,0 sobre a base analítica; valor declarado no material é 25,1%")

assert int(conta_engajamento.sum()) == n_analitica, "faixas de engajamento não fecham na base"
assert int(conta_satisfacao.sum()) == n_analitica, "faixas de satisfação não fecham na base"

print(conta_engajamento.to_string(), "\n")
print(conta_satisfacao.to_string())

engajamento
alto             136
intermediario    539
baixo            673 

satisfacao
alta     418
media    591
baixa    339


## 7. Participação de cada categoria no volume de avaliações

Duas leituras diferentes convivem e precisam estar separadas no gabarito: **quantos produtos** a categoria
tem (portfólio) e **quanto do volume de avaliações** ela concentra. São números distintos — Home & Kitchen
tem mais produtos que Computers & Accessories e menos da metade do volume —, e um cartão que troque um pelo
outro passa despercebido.

T2 comanda o corte: só as **4 categorias com n ≥ 30** são comparadas. As outras 5 somam 7 produtos e entram
em **uma linha única, declarada como suprimida** — não porque sejam irrelevantes, mas porque com 1 ou 2
produtos qualquer percentual é ruído. T6 comanda o rótulo: participação no **volume de avaliações**, jamais
"participação de mercado".

In [7]:
volume_total = float(gold.avaliacoes.sum())
por_categoria = (gold.groupby("categoria_n1")
                     .agg(produtos=("product_id", "size"),
                          volume=("avaliacoes", "sum"),
                          nota_corrigida_mediana=("nota_corrigida", "median"))
                     .sort_values("volume", ascending=False))
por_categoria["participacao_volume_%"] = 100 * por_categoria.volume / volume_total

acima_do_piso = por_categoria[por_categoria.produtos >= 30]
abaixo_do_piso = por_categoria[por_categoria.produtos < 30]

for nome, linha in acima_do_piso.iterrows():
    registrar(f"Produtos em {nome}", int(linha.produtos), "Analítica", n_analitica,
              "contagem de portfólio; categoria acima do piso de 30 (T2)")
    registrar(f"Participação de {nome} no volume de avaliações (%)",
              linha["participacao_volume_%"], "Analítica", int(linha.produtos),
              "DIVIDE(SUM(avaliacoes) da categoria; SUM(avaliacoes) total) · T6: volume, não mercado")
    registrar(f"Nota corrigida mediana em {nome}", linha.nota_corrigida_mediana,
              "Analítica", int(linha.produtos),
              "T1: Wilson, com o n da categoria ao lado; só publicável por estar acima do piso")

registrar("Produtos em categorias abaixo do piso (somadas)", int(abaixo_do_piso.produtos.sum()),
          "Analítica", int(abaixo_do_piso.produtos.sum()),
          "T2: " + str(len(abaixo_do_piso)) + " categorias suprimidas — "
          + ", ".join(f"{c} ({int(v)})" for c, v in abaixo_do_piso.produtos.items())
          + "; nunca ranqueadas nem comparadas")
registrar("Participação das categorias abaixo do piso no volume (%)",
          100 * abaixo_do_piso.volume.sum() / volume_total, "Analítica",
          int(abaixo_do_piso.produtos.sum()),
          "linha única de supressão; declarada no rodapé do visual")
registrar("Categorias de nível 1 acima do piso", len(acima_do_piso), "Analítica", n_analitica,
          "de 9 categorias, 4 alcançam n >= 30 — L3")

subcategorias = gold.categoria_n2.value_counts()
n2_acima = subcategorias[subcategorias >= 30]
registrar("Subcategorias de nível 2 acima do piso", len(n2_acima), "Analítica", n_analitica,
          "unidade de comparação de QA3; T2 aplicada sobre categoria_n2")
registrar("Cobertura das subcategorias acima do piso (%)",
          100 * n2_acima.sum() / n_analitica, "Analítica", int(n2_acima.sum()),
          "parcela do portfólio coberta pelas subcategorias comparáveis")

print(por_categoria.round(2).to_string())

                       produtos      volume  nota_corrigida_mediana  participacao_volume_%
categoria_n1                                                                              
Electronics                 490  14208441.0                    4.08                  59.70
Computers&Accessories       373   6335182.0                    4.15                  26.62
Home&Kitchen                447   2990077.0                    3.97                  12.56
OfficeProducts               31    149675.0                    4.25                   0.63
MusicalInstruments            2     88882.0                    3.88                   0.37
Toys&Games                    1     15867.0                    4.28                   0.07
HomeImprovement               2      8566.0                    4.20                   0.04
Health&PersonalCare           1      3663.0                    3.94                   0.02
Car&Motorbike                 1      1118.0                    3.69                   0.00

## 8. Concentração de Pareto e os dois quadrantes

**Pareto (QA2).** Quantos produtos bastam para juntar 80% do volume de avaliações. A contagem é feita
sobre a base ordenada por volume decrescente, com soma acumulada, e **inclui o produto que cruza o
limiar** — sem essa inclusão a soma para *abaixo* de 80% e a resposta sai menor por um. É a diferença
entre 305 e 306, e é o tipo de divergência que só aparece quando Python e DAX são confrontados.

**Q5 e Q6.** Os dois quadrantes do cruzamento volume × nota. São **listas nominais de produtos**, não
grupos agregados: por T2, o piso de 30 não se aplica a eles, e é por isso que o quadrante de 19 produtos
é publicável. Cada produto exibe seu próprio *n* de avaliações.

In [8]:
n_pareto = int(gold.pareto_80.sum())
registrar("Produtos que concentram 80% do volume de avaliações", n_pareto, "Analítica", n_analitica,
          "ordenado por avaliacoes desc, soma acumulada; INCLUI o produto que cruza o limiar de 80%")
registrar("Participação do grupo de Pareto no portfólio (%)", 100 * n_pareto / n_analitica,
          "Analítica", n_analitica, "n_pareto / base analítica")
registrar("Volume de avaliações concentrado pelo grupo de Pareto (%)",
          100 * gold.loc[gold.pareto_80, "avaliacoes"].sum() / volume_total,
          "Analítica", n_pareto, "confere o corte: tem de ficar imediatamente acima de 80%")

n_q5 = int(gold.q5_risco_vitrine.sum())
n_q6 = int(gold.q6_subexplorado.sum())
registrar("Q5 — alto engajamento e nota abaixo de 4,0", n_q5, "Analítica", n_analitica,
          "lista nominal de produtos; T2 não impõe piso a lista nominal — cada produto exibe seu n")
registrar("Q6 — nota a partir de 4,3 e volume abaixo da mediana", n_q6, "Analítica", n_analitica,
          "funil de Q6; a lista acionável são os 61 que seguem com nota corrigida >= 4,3 (M22, T1). "
           "Rótulo fechado em D-1: bem avaliados com baixo volume de avaliações")
registrar("Volume de avaliações dos produtos de Q5", gold.loc[gold.q5_risco_vitrine, "avaliacoes"].sum(),
          "Analítica", n_q5, "T6: soma de avaliações do quadrante, nunca lida como vendas")

print(f"Pareto 80% : {n_pareto} produtos ({100 * n_pareto / n_analitica:.1f}% da base)")
print(f"Q5         : {n_q5} produtos")
print(f"Q6         : {n_q6} produtos")

Pareto 80% : 306 produtos (22.7% da base)
Q5         : 19 produtos
Q6         : 187 produtos


## 9. Faixas de desconto

A faixa de desconto é o segmentador da página de preço. Entra no gabarito pela mesma razão das outras
contagens: são cinco valores que a medida DAX precisa devolver exatamente, e a soma fecha em 1.348.

A nota corrigida mediana acompanha cada faixa (T1) e todas as cinco estão acima do piso de 30 (T2) —
por isso são comparáveis entre si. Comparáveis, não explicáveis: a associação entre desconto e nota é
tratada no notebook 04, com magnitude e significância juntas (T5), e sem linguagem causal (T4).

In [9]:
ORDEM_FAIXAS = ["0-20%", "21-40%", "41-60%", "61-80%", "81-100%"]
por_faixa = (gold.groupby("faixa_desconto", observed=True)
                 .agg(produtos=("product_id", "size"),
                      nota_corrigida_mediana=("nota_corrigida", "median"),
                      volume_mediano=("avaliacoes", "median"))
                 .reindex(ORDEM_FAIXAS))

for faixa, linha in por_faixa.iterrows():
    registrar(f"Produtos na faixa de desconto {faixa}", int(linha.produtos), "Analítica", n_analitica,
              "faixa fechada à direita; soma das 5 faixas fecha na base analítica")
    registrar(f"Nota corrigida mediana na faixa {faixa}", linha.nota_corrigida_mediana,
              "Analítica", int(linha.produtos),
              "T1: Wilson com o n ao lado; faixa acima do piso de 30 (T2)"
              if linha.produtos >= 30 else "ABAIXO do piso de 30 — não comparar (T2)")

assert int(por_faixa.produtos.sum()) == n_analitica, "faixas de desconto não fecham na base"
print(por_faixa.round(3).to_string())

                produtos  nota_corrigida_mediana  volume_mediano
faixa_desconto                                                  
0-20%                175                   4.136          6736.0
21-40%               329                   4.083          7241.0
41-60%               469                   4.025          3025.0
61-80%               324                   4.035          4147.0
81-100%               51                   3.906          4426.0


## 10. Dimensão do corpus textual

O dashboard exibe a contagem de avaliações analisadas, e essa contagem é a que mais facilmente sai errada:
a mesma avaliação aparece referenciada em mais de um produto. Contar referências dá **10.588**; contar
avaliações distintas dá **9.255**. A diferença — 1.333 — é 14% de inflação em um número que a banca
consegue conferir.

Por T8, três coisas andam juntas e nenhuma pode ser omitida: a **unidade** (review individual, não bloco),
a **deduplicação por `review_id`** e o **truncamento em 8 avaliações por produto** (L1). O truncamento é o
que impede a frase "a maioria dos clientes diz…": o texto disponível é o recorte que a vitrine exibe, não
uma amostra do que foi escrito.

In [10]:
# Referências = bloco de reviews explodido; medido sobre os produtos da base analítica.
blocos = silver[silver.product_id.isin(gold.product_id)].copy()
explodido = blocos.assign(_ref=blocos.review_id.str.split(",")).explode("_ref")
explodido["_ref"] = explodido._ref.str.strip()

n_referencias = len(explodido)
n_distintos = int(explodido._ref.nunique())
n_repetidas = n_referencias - n_distintos
por_produto = explodido.groupby("product_id")["_ref"].size()

registrar("Referências de avaliação (bloco explodido)", n_referencias, "Analítica", n_analitica,
          "T8: unidade = referência, ANTES de deduplicar; não é o número a publicar")
registrar("Avaliações distintas após deduplicação por review_id", n_distintos, "Corpus de avaliações",
          n_distintos, "T8: este é o número publicável; bate com 00_data/gold/reviews.csv")
registrar("Referências repetidas entre produtos", n_repetidas, "Analítica", n_analitica,
          "diferença entre referência e review individual; 14% de inflação se ignorada")
registrar("Avaliações por produto — média (truncamento)", por_produto.mean(), "Analítica", n_analitica,
          "L1: a origem trunca em 8 avaliações por produto; a média medida é 7,85")
registrar("Avaliações por produto — máximo (truncamento)", por_produto.max(), "Analítica", n_analitica,
          "L1: teto rígido de 8; nenhum produto traz mais do que isso")

assert n_distintos == n_reviews, "corpus do notebook diverge de 00_data/gold/reviews.csv"
print(f"referências : {n_referencias}")
print(f"distintas   : {n_distintos}")
print(f"repetidas   : {n_repetidas}")
print(f"por produto : média {por_produto.mean():.2f} · máximo {por_produto.max()}")

referências : 10588
distintas   : 9255
repetidas   : 1333
por produto : média 7.85 · máximo 8


## 11. Exportação do gabarito

`tab12_valores_referencia.csv` é o único arquivo deste notebook que o resto do projeto consome. Sem ele
não há como reconciliar Python e Power BI: cada linha é uma medida do dashboard, com o valor que ela tem
de devolver, a base em que foi contada e o *n* efetivo.

Duas travas antes de escrever: **nenhuma linha sem base ou sem *n*** (T3) e **nenhum nome de medida
repetido** — nome duplicado com valores diferentes transformaria o gabarito no problema que ele existe
para resolver.

In [11]:
gabarito = pd.DataFrame(REFERENCIA, columns=["medida", "valor", "base", "n", "observacao"])

duplicadas = gabarito.medida[gabarito.medida.duplicated()].tolist()
assert not duplicadas, f"medidas com nome repetido no gabarito: {duplicadas}"
assert gabarito.base.notna().all() and gabarito.n.notna().all(), "linha sem base ou sem n — T3"
assert not gabarito.medida.str.contains("venda|receita|demanda|popularidade", case=False).any(), \
    "T6: rótulo de medida usa termo vedado para rating_count"

destino = os.path.join(TAB, "tab12_valores_referencia.csv")
gabarito.to_csv(destino, index=False, encoding="utf-8")
print(f"{len(gabarito)} medidas -> {os.path.relpath(destino, RAIZ)}")
gabarito

64 medidas -> 02_results/tables/tab12_valores_referencia.csv


,medida,valor,base,n,observacao
0,Contagem de linhas — base bruta,1465.0000,Bruta,1465,"linhas da origem, antes da deduplicação; serve..."
1,Contagem de produtos — base de produtos,1351.0000,Produtos,1351,DISTINCTCOUNT(product_id) após a regra de dedu...
2,Contagem de produtos — nota válida,1350.0000,Nota válida,1350,produtos com rating convertível; 1 registro co...
3,Contagem de produtos — engajamento válido,1349.0000,Engajamento válido,1349,produtos com rating_count convertível; 2 nulos...
4,Contagem de produtos — base analítica,1348.0000,Analítica,1348,padrão do dashboard: nota e volume de avaliaçõ...
...,...,...,...,...,...
59,Referências de avaliação (bloco explodido),10588.0000,Analítica,1348,"T8: unidade = referência, ANTES de deduplicar;..."
60,Avaliações distintas após deduplicação por rev...,9255.0000,Corpus de avaliações,9255,T8: este é o número publicável; bate com 00_da...
61,Referências repetidas entre produtos,1333.0000,Analítica,1348,diferença entre referência e review individual...
62,Avaliações por produto — média (truncamento),7.8546,Analítica,1348,L1: a origem trunca em 8 avaliações por produt...


## 12. Figuras de conferência

Duas figuras, ambas na paleta v06 aplicada por módulo (T9) e com fonte, base e *n* no rodapé (T3).
Numeração a partir de **fig16**, com o sufixo `_03_` do notebook: fig01–fig13 vêm do material anterior e
fig14–fig15 do notebook 02. Nenhuma delas é sobrescrita.

A primeira confere as duas segmentações categóricas; a segunda, a participação no volume de avaliações,
com as categorias abaixo do piso reunidas em uma barra declarada (T2). Nenhuma das duas passa de 5 séries.

In [12]:
# T3 — fonte, base e n embaixo da figura, nunca sobre o eixo.
def salvar(fig, nome):
    confere_visual(fig, nome)          # T3 e T9 conferidos na figura, não no código
    fig.savefig(os.path.join(IMG, nome), dpi=150, bbox_inches="tight",
                pad_inches=0.28, facecolor=BASE["canvas"])
    plt.close(fig)
    print(f"  {nome}")


checar_series(3)  # engajamento e satisfação têm 3 faixas cada

fig, eixos = plt.subplots(1, 2, figsize=(10, 4))
for eixo, serie, cores, titulo in [
    (eixos[0], conta_engajamento, [AMBAR, CLARA, AZUL], "Engajamento — volume de avaliações"),
    (eixos[1], conta_satisfacao, [TEAL, CLARA, LARANJA], "Satisfação — nota"),
]:
    barras = eixo.bar(serie.index, serie.values, color=cores, width=0.62)
    eixo.set_title(titulo)
    eixo.set_ylabel("produtos")
    eixo.set_ylim(0, serie.max() * 1.18)
    eixo.grid(axis="x", visible=False)
    for barra, valor in zip(barras, serie.values):
        eixo.text(barra.get_x() + barra.get_width() / 2, valor + serie.max() * 0.03,
                  f"{valor}", ha="center", fontsize=9, color=TINTA)

fig.suptitle("Contagens que o DAX tem de reproduzir", color=TINTA, fontsize=13)
fig.tight_layout(rect=[0, 0.09, 1, 0.97])
p90_txt = f"{p90_avaliacoes:,.0f}".replace(",", ".")
mediana_txt = f"{mediana_avaliacoes:,.0f}".replace(",", ".")
rodape(fig, "Fonte: 00_data/gold/base_analitica.csv · base analítica, n = 1.348 · "
            f"engajamento alto >= p90 ({p90_txt}), baixo < mediana ({mediana_txt}) · "
            "satisfação baixa < 4,0 e alta >= 4,3 · rating_count é volume de avaliações (T6)")
salvar(fig, "fig_med_01_painel_medidas.png")

  fig_med_01_painel_medidas.png

In [13]:
rotulos = list(acima_do_piso.index) + [f"{len(abaixo_do_piso)} categorias\nabaixo do piso"]
valores = list(acima_do_piso["participacao_volume_%"]) + [100 * abaixo_do_piso.volume.sum() / volume_total]
contagens = list(acima_do_piso.produtos) + [int(abaixo_do_piso.produtos.sum())]
cores = [AMBAR, AZUL, TEAL, LARANJA, ESTRUTURA["grade"]]  # grade reservada à linha suprimida
checar_series(len(rotulos))

fig, eixo = plt.subplots(figsize=(9, 4.4))
barras = eixo.barh(rotulos[::-1], valores[::-1], color=cores[::-1], height=0.6)
eixo.set_xlabel("participação no volume de avaliações (%)")
eixo.set_xlim(0, max(valores) * 1.55)
eixo.grid(axis="y", visible=False)
# L8 — a leitura de participação por categoria declara as DUAS versões, com e sem as
# listagens variantes. Colapsar cada grupo que repete o par (avaliacoes, nota) dentro
# da categoria mostra quanto do volume é o mesmo contador visto mais de uma vez: em
# Electronics, 59,7% caem para 53,1%. Publicar só a primeira deixaria o leitor somar
# avaliações que são a mesma avaliação.
colapsada = gold.drop_duplicates(subset=["categoria_n1", "avaliacoes", "nota"])
volume_colapsado = colapsada.avaliacoes.sum()
sem_variantes = {cat: 100 * colapsada.loc[colapsada.categoria_n1 == cat, "avaliacoes"].sum() / volume_colapsado
                 for cat in acima_do_piso.index}
sem_variantes[rotulos[-1]] = 100 * colapsada.loc[
    ~colapsada.categoria_n1.isin(acima_do_piso.index), "avaliacoes"].sum() / volume_colapsado

for barra, rotulo, valor, n_cat in zip(barras, rotulos[::-1], valores[::-1], contagens[::-1]):
    eixo.text(valor + max(valores) * 0.015, barra.get_y() + barra.get_height() / 2,
              f"{valor:.1f}%  ·  {sem_variantes[rotulo]:.1f}% sem variantes  ·  n = {n_cat}".replace(".", ","),
              va="center", fontsize=8.4, color=TINTA)

eixo.set_title("Participação de cada categoria no volume de avaliações — com e sem listagens variantes")
fig.tight_layout(rect=[0, 0.12, 1, 1])
rodape(fig, "Fonte: 00_data/gold/base_analitica.csv · base analítica, n = 1.348 · "
            "volume total = 23.801.471 avaliações · piso amostral n >= 30 (T2): 5 categorias somando "
            "7 produtos aparecem agregadas e não são ranqueadas · volume de avaliações, não vendas (T6) · "
            "cada barra traz também a versão que colapsa as listagens variantes do mesmo item, "
            "obrigatória em toda leitura de participação (L8)")
salvar(fig, "fig_med_02_participacao_categoria.png")

  fig_med_02_participacao_categoria.png


## 13. Validação final — reconciliação com a base factual

O notebook só vale se os números que ele produz forem os mesmos do material. Esta célula confronta,
um a um, os valores calculados aqui com `method/definitions/base_factual_e_bases_de_calculo.md`. Se
qualquer linha divergir, a célula falha: um gabarito que discorda da fonte é pior do que gabarito
nenhum, porque dá confiança a um número errado.

Os valores esperados estão escritos literalmente, e não lidos de outro arquivo gerado — a conferência
precisa ser contra o documento, não contra o mesmo código que produziu o número.

In [14]:
esperado = [
    ("Base bruta",                          n_bruta,                                    1465),
    ("Base de produtos",                    n_produtos,                                 1351),
    ("Base nota válida",                    n_nota_valida,                              1350),
    ("Base engajamento válido",             n_engaj_valido,                             1349),
    ("Base analítica",                      n_analitica,                                1348),
    ("p90 de avaliações",                   p90_avaliacoes,                             36384),
    ("Mediana de avaliações",               mediana_avaliacoes,                         4740),
    ("Satisfação baixa (nota < 4,0)",       int(conta_satisfacao["baixa"]),             339),
    ("Satisfação baixa (%)",                round(100 * conta_satisfacao["baixa"] / n_analitica, 1), 25.1),
    ("Satisfação alta (nota >= 4,3)",       int(conta_satisfacao["alta"]),              418),
    ("Q5 — risco de vitrine",               n_q5,                                       19),
    ("Q6 — nota alta, volume baixo",        n_q6,                                       187),
    ("Pareto 80%",                          n_pareto,                                   306),
    ("Pareto 80% (% da base)",              round(100 * n_pareto / n_analitica, 1),     22.7),
    ("Categorias n1 acima do piso",         len(acima_do_piso),                         4),
    ("Electronics",                         int(por_categoria.loc["Electronics", "produtos"]),            490),
    ("Home&Kitchen",                        int(por_categoria.loc["Home&Kitchen", "produtos"]),           447),
    ("Computers&Accessories",               int(por_categoria.loc["Computers&Accessories", "produtos"]),  373),
    ("OfficeProducts",                      int(por_categoria.loc["OfficeProducts", "produtos"]),         31),
    ("Categorias abaixo do piso (produtos)", int(abaixo_do_piso.produtos.sum()),        7),
    ("Subcategorias n2 acima do piso",      len(n2_acima),                              8),
    ("Cobertura das subcategorias (%)",     round(100 * n2_acima.sum() / n_analitica, 1), 88.3),
    ("Preço de tabela mediano",             gold.preco_tab.median(),                    1795),
    ("Preço com desconto mediano",          gold.preco_desc.median(),                   899),
    ("Desconto mediano (%)",                gold.desconto.median(),                     49),
    ("Nota mediana",                        gold.nota.median(),                         4.10),
    ("Referências de avaliação",            n_referencias,                              10588),
    ("Avaliações distintas (review_id)",    n_distintos,                                9255),
    ("Referências repetidas",               n_repetidas,                                1333),
    ("Truncamento — média por produto",     round(por_produto.mean(), 2),               7.85),
    ("Truncamento — máximo por produto",    int(por_produto.max()),                     8),
]

# "item" não pode virar nome de coluna: colide com Series.item e quebra a formatação.
conferencia = pd.DataFrame(esperado, columns=["indicador", "obtido", "material"])
conferencia["confere"] = np.isclose(conferencia.obtido.astype(float),
                                    conferencia.material.astype(float), atol=0.051)

print("RECONCILIAÇÃO COM A BASE FACTUAL")
print("-" * 74)
for _, linha in conferencia.iterrows():
    marca = "ok " if linha.confere else "FALHA"
    print(f"  {marca} {linha.indicador:<38} obtido {linha.obtido:>12} · material {linha.material:>9}")
print("-" * 74)
print(f"Reconciliação das bases: {n_bruta} -> {n_produtos} -> {n_nota_valida} -> "
      f"{n_engaj_valido} -> {n_analitica}")
print(f"Soma das categorias: {int(acima_do_piso.produtos.sum())} (acima do piso) + "
      f"{int(abaixo_do_piso.produtos.sum())} (suprimidas) = {int(por_categoria.produtos.sum())}")
print(f"Gabarito exportado: {len(gabarito)} medidas em 02_results/tables/tab12_valores_referencia.csv")
print("Figuras: fig_med_01_painel_medidas.png · fig_med_02_participacao_categoria.png "
      "(fig01–fig15, dos notebooks anteriores, intactas)")

falhas = conferencia[~conferencia.confere]
assert falhas.empty, f"divergência contra o material:\n{falhas.to_string(index=False)}"
print("\nTodos os 31 itens conferem com method/definitions/base_factual_e_bases_de_calculo.md.")

RECONCILIAÇÃO COM A BASE FACTUAL
--------------------------------------------------------------------------
  ok  Base bruta                             obtido       1465.0 · material    1465.0
  ok  Base de produtos                       obtido       1351.0 · material    1351.0
  ok  Base nota válida                       obtido       1350.0 · material    1350.0
  ok  Base engajamento válido                obtido       1349.0 · material    1349.0
  ok  Base analítica                         obtido       1348.0 · material    1348.0
  ok  p90 de avaliações                      obtido      36384.0 · material   36384.0
  ok  Mediana de avaliações                  obtido       4740.0 · material    4740.0
  ok  Satisfação baixa (nota < 4,0)          obtido        339.0 · material     339.0
  ok  Satisfação baixa (%)                   obtido         25.1 · material      25.1
  ok  Satisfação alta (nota >= 4,3)          obtido        418.0 · material     418.0
  ok  Q5 — risco de vitrine     

## 14. As três decisões que pendiam deste gabarito, e como fecharam

- **D-1**, fechada em 24/09/2026 — visibilidade real (impressão, clique) não existe na base. Vale
  volume de avaliações abaixo da mediana como **proxy declarado de exposição**: Q6 é o **funil de
  187**, e a lista acionável são os **61** que seguem com nota corrigida ≥ 4,3. O rótulo publicado é
  *bem avaliados com baixo volume de avaliações*, nunca "baixa visibilidade" (T6).
- **D-3**, fechada em 01/10/2026 — a `nota_corrigida` é o **limite inferior de Wilson a 95%**, e é
  assim que ela entra em todo ranking (T1). A média bayesiana foi descartada por exigir arbitrar um
  *prior*.
- **D-7**, fechada em 01/10/2026 — o catálogo tem **22 medidas (M01–M22)**, em
  `03_dashboard/medidas_dax.md`. Este gabarito é o valor de referência que cada uma precisa reproduzir.